In [ ]:
# Import required libraries
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from transformers import RobertaTokenizer
from transformers import AutoTokenizer

from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

from tqdm import tqdm

import os
import certifi

os.environ['SSL_CERT_FILE'] = certifi.where()

In [ ]:
# Load the dataset from CSV file
data = pd.read_csv("Twitter_Data.csv")

# Display first 5 rows
print(data.head())

# Display dataset information
print(data.info())

# Check missing values
print(data.isnull().sum())

In [ ]:
# Display shape of dataset
print("Dataset Shape:", data.shape)

# Display column names
print("Columns:", data.columns)

# Summary statistics
print(data.describe())

# Count of each sentiment category
print(data['category'].value_counts())

# Percentage distribution
print(data['category'].value_counts(normalize=True) * 100)

# Check duplicate rows
print("Duplicate Rows:", data.duplicated().sum())

# Display random samples
print(data.sample(5))

In [ ]:
# Download stopwords
nltk.download('stopwords')

# Initialize stopwords
stop_words = set(stopwords.words('english'))

# Remove negation words from stopwords
negation_words = {'not', 'no', 'nor', 'never'}
stop_words = stop_words - negation_words

# Text cleaning function
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords
    text = " ".join([word for word in text.split() if word not in stop_words])

    return text

# Apply cleaning
data['clean_text'] = data['clean_text'].astype(str).apply(clean_text)

# Remove missing values
data = data.dropna()

# Reset index
data = data.reset_index(drop=True)

# Convert labels from (-1,0,1) to (0,1,2)
data['category'] = data['category'] + 1

# Convert labels to integer
data['category'] = data['category'].astype(int)

# Display cleaned data
print(data.head())

In [ ]:
# Calculate text length
data['text_length'] = data['clean_text'].apply(lambda x: len(x.split()))

plt.figure()
plt.hist(data['text_length'], bins=30)

plt.title("Text Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

In [ ]:
avg_length = data.groupby('category')['text_length'].mean()

labels = ['Negative', 'Neutral', 'Positive']

plt.figure()
bars = plt.bar(labels, avg_length)

# Add values
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval,1),
             ha='center', va='bottom')

plt.title("Average Text Length per Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Average Words")

plt.show()

In [ ]:
# ===== Split Dataset into Train, Validation, and Test =====
# Data is divided into training, validation, and testing sets

from sklearn.model_selection import train_test_split


# ===== First Split (Train and Temporary Set) =====
# 80% data is used for training
# 20% data is kept for validation and testing

train_df, temp_df = train_test_split(
    data,
    test_size=0.2,                  # 20% for validation + test
    stratify=data['category'],      # maintain class distribution
    random_state=42                 # ensure reproducibility
)


# ===== Second Split (Validation and Test) =====
# Temporary data is split equally into validation and test sets
# Final split: 80% train, 10% validation, 10% test

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,                  # split remaining 20% into 10% + 10%
    stratify=temp_df['category'],   # maintain class distribution
    random_state=42
)

In [ ]:

import os

# ---- Install dependencies ----
!pip install -q transformers torch tqdm certifi

# ---- Download models from HuggingFace Hub (replaces local paths) ----
DISTILBERT_PATH = "distilbert-base-uncased"
ROBERTA_PATH    = "roberta-base"

print("✅ Setup complete")

In [ ]:
# ===== Load DistilBERT Tokenizer =====
# Pre-trained tokenizer is used to convert text into numerical format
tokenizer = DistilBertTokenizer.from_pretrained(DISTILBERT_PATH)

In [ ]:


# ===== Encode Training, Validation, and Test Data =====
# Text is converted into token IDs and attention masks

train_enc = tokenizer(
    train_df['clean_text'].tolist(),   # training text
    padding=True,                      # pad sequences to same length
    truncation=True,                   # truncate long sequences
    max_length=32,                     # maximum sequence length
    return_tensors='pt'                # return PyTorch tensors
)

val_enc = tokenizer(
    val_df['clean_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors='pt'
)

test_enc = tokenizer(
    test_df['clean_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors='pt'
)


# ===== Extract Input IDs and Attention Masks =====
# Input IDs represent tokenised words
# Attention masks indicate real tokens (1) and padding (0)

train_inputs = train_enc['input_ids']
train_masks  = train_enc['attention_mask']

val_inputs = val_enc['input_ids']
val_masks  = val_enc['attention_mask']

test_inputs = test_enc['input_ids']
test_masks  = test_enc['attention_mask']


# ===== Extract Labels =====
# Labels are taken from dataset for training and evaluation

train_labels = train_df['category'].values
val_labels   = val_df['category'].values
test_labels  = test_df['category'].values


# ===== Display Shapes =====
# Shapes are printed to verify correct encoding

print("Train masks shape:", train_masks.shape)
print("Validation masks shape:", val_masks.shape)
print("Test masks shape:", test_masks.shape)

In [ ]:
# Load pre-trained DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained(DISTILBERT_PATH, num_labels=3)

print("DistilBERT model loaded successfully")

In [ ]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model.to(device)

print("Device used:", device)

In [ ]:
# Convert labels to tensor
train_labels = torch.tensor(train_labels, dtype=torch.long)
val_labels = torch.tensor(val_labels, dtype=torch.long)

# Create training and validation datasets
train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
val_dataset = TensorDataset(val_inputs, val_masks, val_labels)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

# Display number of batches
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
epochs = 5

optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        batch = tuple(t.to(device) for t in batch)
        input_ids, attention_mask, labels = batch

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()

        # Gradient clipping
        clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    avg_train_loss = total_loss / len(train_loader)

    # ===== VALIDATION LOOP =====
    model.eval()
    val_loss = 0
    preds, true = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = tuple(t.to(device) for t in batch)
            input_ids, attention_mask, labels = batch

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()

            logits = outputs.logits
            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(true, preds)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {avg_train_loss}")
    print(f"Val Loss: {avg_val_loss}")
    print(f"Val Accuracy: {val_acc}")

In [ ]:
# Plot Training vs Validation
plt.figure()
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Training vs Validation Loss")
plt.savefig("loss_curve.png", dpi=400)
plt.show()

In [ ]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:

        # Move batch to device
        batch = tuple(t.to(device) for t in batch)
        input_ids, attention_mask, labels = batch

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Get predicted class
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Accuracy
val_accuracy = accuracy_score(true_labels, predictions)
print("Validation Accuracy:", val_accuracy)

# Detailed report
print("\nClassification Report:\n")
print(classification_report(true_labels, predictions))

In [ ]:
# Convert test labels to tensor
test_labels = torch.tensor(test_labels, dtype=torch.long)

# Create test dataset
test_dataset = TensorDataset(test_inputs, test_masks, test_labels)

# Create test DataLoader
test_loader = DataLoader(test_dataset, batch_size=64)

print("Test batches:", len(test_loader))

In [ ]:
model.eval()

test_predictions = []
test_true_labels = []

with torch.no_grad():
   for batch in test_loader:

        # Move batch to device
        batch = tuple(t.to(device) for t in batch)
        input_ids, attention_mask, labels = batch

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Get predicted class
        preds = torch.argmax(logits, dim=1)

        test_predictions.extend(preds.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

print("\n===== DistilBERT Results =====")
# Accuracy
distilbert_accuracy = accuracy_score(test_true_labels, test_predictions)
print("DistilBERT Accuracy:", distilbert_accuracy)

# Detailed report
print("\nClassification Report:\n")
print(classification_report(test_true_labels, test_predictions))

In [ ]:
# ===== Calculate Precision =====
# Precision measures how many predicted positives are correct

precision = precision_score(
    test_true_labels,        # actual labels
    test_predictions,        # predicted labels
    average='weighted'       # weighted average for all classes
)


# ===== Calculate Recall =====
# Recall measures how many actual positives are correctly identified

recall = recall_score(
    test_true_labels,
    test_predictions,
    average='weighted'
)


# ===== Calculate F1 Score =====
# F1 score is the balance between precision and recall

f1 = f1_score(
    test_true_labels,
    test_predictions,
    average='weighted'
)


# ===== Display Evaluation Results =====
# All performance metrics are printed

print("Accuracy:", distilbert_accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
# ===== Load RoBERTa Tokenizer (Local) =====
from transformers import AutoTokenizer

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)


# ===== Reduce Dataset Size (IMPORTANT FOR SPEED) =====
# Balanced sampling is used for faster training

train_df_rb = train_df.sample(min(30000, len(train_df)), random_state=42)
val_df_rb   = val_df.sample(min(5000, len(val_df)), random_state=42)
test_df_rb  = test_df.sample(min(5000, len(test_df)), random_state=42)

print(f"Train RoBERTa size : {len(train_df_rb)}")
print(f"Val   RoBERTa size : {len(val_df_rb)}")
print(f"Test  RoBERTa size : {len(test_df_rb)}")


# ===== Encode Training Data =====
train_enc_rb = roberta_tokenizer(
    train_df_rb['clean_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=24,           # reduced from 32 → faster
    return_tensors='pt'
)


# ===== Encode Validation Data =====
val_enc_rb = roberta_tokenizer(
    val_df_rb['clean_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=24,
    return_tensors='pt'
)


# ===== Encode Test Data =====
test_enc_rb = roberta_tokenizer(
    test_df_rb['clean_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=24,
    return_tensors='pt'
)


# ===== Extract Inputs and Attention Masks =====

train_inputs_rb = train_enc_rb['input_ids']
train_masks_rb  = train_enc_rb['attention_mask']

val_inputs_rb = val_enc_rb['input_ids']
val_masks_rb  = val_enc_rb['attention_mask']

test_inputs_rb = test_enc_rb['input_ids']
test_masks_rb  = test_enc_rb['attention_mask']


# ===== Convert Labels to Tensor Format =====

train_labels_rb = torch.tensor(train_df_rb['category'].values, dtype=torch.long)
val_labels_rb   = torch.tensor(val_df_rb['category'].values, dtype=torch.long)
test_labels_rb  = torch.tensor(test_df_rb['category'].values, dtype=torch.long)

In [ ]:
# ===== Create TensorDatasets =====
# Inputs, attention masks, and labels are combined into dataset format

train_dataset_rb = TensorDataset(
    train_inputs_rb,
    train_masks_rb,
    train_labels_rb
)

val_dataset_rb = TensorDataset(
    val_inputs_rb,
    val_masks_rb,
    val_labels_rb
)

test_dataset_rb = TensorDataset(
    test_inputs_rb,
    test_masks_rb,
    test_labels_rb
)


# ===== Create DataLoaders =====
# Larger batch size is used for faster training

train_loader_rb = DataLoader(
    train_dataset_rb,
    batch_size=128,     # increased from 64 → faster
    shuffle=True
)

val_loader_rb = DataLoader(
    val_dataset_rb,
    batch_size=128      # match train batch size
)

test_loader_rb = DataLoader(
    test_dataset_rb,
    batch_size=128      # match for consistency
)

In [ ]:
# ===== Import RoBERTa Model =====
from transformers import AutoModelForSequenceClassification


# ===== Load Pre-trained RoBERTa Model =====
model_rb = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_PATH,        # ← must be this, not roberta_path
    num_labels=3
)

# ===== Freeze Base RoBERTa Layers =====
# Only classification head will be trained (faster + sufficient)

for param in model_rb.roberta.parameters():
    param.requires_grad = False


# ===== Move Model to Device =====
model_rb.to(device)


# ===== Define Optimizer =====
# Only trainable parameters are passed (important)

optimizer_rb = AdamW(
    filter(lambda p: p.requires_grad, model_rb.parameters()),
    lr=3e-5     # slightly higher since fewer parameters are trained
)


# ===== Calculate Total Training Steps =====
total_steps_rb = len(train_loader_rb) * epochs


# ===== Define Learning Rate Scheduler =====
scheduler_rb = get_linear_schedule_with_warmup(
    optimizer_rb,
    num_warmup_steps=int(0.1 * total_steps_rb),   # small warm-up added
    num_training_steps=total_steps_rb
)

In [ ]:
# ===== Store Training History =====
history_rb = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": []
}


# ===== Training Loop =====
for epoch in range(epochs):
    print(f"\nRoBERTa Epoch {epoch+1}/{epochs}")

    model_rb.train()
    total_loss = 0



    for batch in tqdm(train_loader_rb):

        batch = tuple(t.to(device) for t in batch)
        input_ids, attention_mask, labels = batch

        optimizer_rb.zero_grad()

        # ===== Mixed Precision (FASTER) =====
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model_rb(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        total_loss += loss.item()

        # ===== Backpropagation =====
        loss.backward()

        # Gradient clipping
        clip_grad_norm_(model_rb.parameters(), 1.0)

        optimizer_rb.step()
        scheduler_rb.step()

    avg_train_loss = total_loss / len(train_loader_rb)


    # ===== Validation =====
    model_rb.eval()
    val_loss = 0
    preds, true = [], []

    with torch.no_grad():
        for batch in val_loader_rb:

            batch = tuple(t.to(device) for t in batch)
            input_ids, attention_mask, labels = batch

            outputs = model_rb(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()

            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true.extend(labels.cpu().numpy())


    avg_val_loss = val_loss / len(val_loader_rb)
    val_acc = accuracy_score(true, preds)

    history_rb["train_loss"].append(avg_train_loss)
    history_rb["val_loss"].append(avg_val_loss)
    history_rb["val_acc"].append(val_acc)

    print("Train Loss:", avg_train_loss)
    print("Val Loss:", avg_val_loss)
    print("Val Accuracy:", val_acc)

In [ ]:
# ===== Set Model to Evaluation Mode =====
# Dropout and training layers are disabled during testing

model_rb.eval()


# ===== Initialize Lists for Predictions =====
# These lists will store predicted labels and true labels

test_preds_rb = []
test_true_rb = []


# ===== Testing Loop =====
# Model is evaluated on unseen test data

with torch.no_grad():   # No gradient calculation (faster and memory efficient)

    for batch in test_loader_rb:

        # Move batch data to device (GPU or CPU)
        batch = tuple(t.to(device) for t in batch)
        input_ids, attention_mask, labels = batch

        # Forward pass (prediction only, no loss needed here)
        outputs = model_rb(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract logits (raw model outputs)
        logits = outputs.logits

        # Convert logits to predicted class labels
        preds = torch.argmax(logits, dim=1)

        # Store predictions and true labels
        test_preds_rb.extend(preds.cpu().numpy())
        test_true_rb.extend(labels.cpu().numpy())


# ===== Display Results =====
# Accuracy and classification report are calculated

print("\n===== RoBERTa Results =====")

# Calculate accuracy
roberta_accuracy = accuracy_score(test_true_rb, test_preds_rb)
print("RoBERTa Accuracy:", roberta_accuracy)

# Display precision, recall, and F1-score
print(classification_report(test_true_rb, test_preds_rb))

In [ ]:
# ===== Convert to numpy (IMPORTANT FIX) =====
test_true_rb = np.array(test_true_rb)
test_preds_rb = np.array(test_preds_rb)


# ===== Compute Confusion Matrix =====
cm_rb = confusion_matrix(test_true_rb, test_preds_rb)


# ===== Normalize Confusion Matrix =====
cm_rb_norm = cm_rb.astype('float') / cm_rb.sum(axis=1, keepdims=True)


# ===== Define Proper Labels =====
class_names = ['Negative', 'Neutral', 'Positive']


# ===== Plot =====
plt.figure(figsize=(6,5))

sns.heatmap(
    cm_rb_norm,
    annot=True,
    fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="coolwarm"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("RoBERTa Confusion Matrix")

plt.savefig("cm_roberta.png", dpi=400)
plt.show()

In [ ]:
# ===== Correct Class Labels =====
class_names = ['Negative', 'Neutral', 'Positive']


# ===== Count Sentiment Categories =====
counts = data['category'].value_counts().sort_index()


# ===== Create Bar Plot =====}
plt.figure()
bars = plt.bar(class_names, counts.values)


# ===== Add Values on Bars =====
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        yval,
        int(yval),
        ha='center',
        va='bottom'
    )


# ===== Labels =====
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

plt.show()

In [ ]:
# ===== Define Correct Class Names =====
class_names = ['Negative', 'Neutral', 'Positive']


# ===== Count Predicted Sentiments =====
# Ensure all 3 classes are counted

pred_counts = np.bincount(test_predictions, minlength=3)


# ===== Create Bar Plot =====
plt.figure()
bars = plt.bar(class_names, pred_counts)


# ===== Add Values on Bars =====
for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        yval,
        int(yval),
        ha='center',
        va='bottom'
    )


# ===== Labels =====
plt.title("Predicted Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

plt.show()

In [ ]:
# ===== Convert to numpy =====
test_true_labels = np.array(test_true_labels)
test_predictions = np.array(test_predictions)


# ===== Compute Confusion Matrix =====
cm = confusion_matrix(test_true_labels, test_predictions)


# ===== Normalize =====
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)


# ===== Correct Class Names =====
class_names = ['Negative', 'Neutral', 'Positive']


# ===== Plot =====
plt.figure(figsize=(6,5))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="coolwarm"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("DistilBERT Confusion Matrix")

plt.savefig("cm_distilbert.png", dpi=400)
plt.show()

In [ ]:
# ===== Create Pipeline =====
# TF-IDF vectorisation and Naive Bayes model are combined

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,     # limit number of features
        ngram_range=(1,2)      # use unigrams and bigrams
    )),
    ('nb', MultinomialNB())    # Naive Bayes classifier
])


# ===== Cross-Validation Setup =====
# 5-fold stratified cross-validation is used for reliable evaluation

cv = StratifiedKFold(
    n_splits=5,               # number of folds
    shuffle=True,             # shuffle data before splitting
    random_state=42           # for reproducibility
)


# ===== Perform Cross-Validation =====
# Model is evaluated on different splits of data

scores = cross_val_score(
    pipeline,
    data['clean_text'],       # input text
    data['category'],         # labels
    cv=cv
)


# ===== Display Cross-Validation Accuracy =====
print("\n===== Naive Bayes Results =====")

nb_accuracy = scores.mean()   # average accuracy across folds
print("Naive Bayes Accuracy:", nb_accuracy)


# ===== Train Model on Training Data =====
# Final model is trained using the same split as transformer models

pipeline.fit(
    train_df['clean_text'],
    train_df['category']
)


# ===== Make Predictions on Test Data =====
# Predictions are generated for unseen test data

y_pred = pipeline.predict(test_df['clean_text'])


# ===== Display Classification Report =====
# Precision, recall, and F1-score are shown

print(classification_report(test_df['category'], y_pred))

In [ ]:
# ===== Convert to plain numpy arrays (removes CUDA tensor formatting) =====
y_pred_np      = np.array([p.item() if hasattr(p, 'item') else p for p in y_pred])
true_labels_np = np.array(test_df['category'].values)

# Human-readable axis labels
class_names = ['Negative', 'Neutral', 'Positive']

# ===== Compute Confusion Matrix =====
cm_nb = confusion_matrix(true_labels_np, y_pred_np)

# ===== Normalize =====
cm_nb_norm = cm_nb.astype('float') / cm_nb.sum(axis=1)[:, np.newaxis]

# ===== Plot =====
plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_nb_norm,
    annot=True,
    fmt=".2f",
    xticklabels=class_names,   # ← plain strings, not tensors
    yticklabels=class_names,   # ← plain strings, not tensors
    cmap="coolwarm"
)

plt.title("Naive Bayes Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("cm_nb.png", dpi=400)
plt.show()

In [ ]:
# ===== Error Analysis — Naive Bayes Misclassification =====
test_df_reset = test_df.reset_index(drop=True)

# Convert predictions to plain numpy array
y_pred_np = np.array([p.item() if hasattr(p, 'item') else p for p in y_pred])

# Build comparison table
errors = pd.DataFrame({
    "text" : test_df_reset['clean_text'].values,
    "true" : test_df_reset['category'].values,
    "pred" : y_pred_np
})

# Filter wrong predictions only
misclassified = errors[errors['true'] != errors['pred']]

# ===== Print Summary =====
print("===== Misclassification Analysis =====")
print(f"Correctly classified : {len(errors) - len(misclassified)} / {len(errors)}")
print(f"Misclassified        : {len(misclassified)} / {len(errors)}")
print()
print("Label Guide → 0=Negative | 1=Neutral | 2=Positive")
print()

# ===== Display as clean table (matches your original format) =====
print(misclassified.head(10).to_string(index=False))